# Checklist Tool

In this notebook we will look at providing our agent with a tool that allows it create and manage a check list. This can be used of tracking progress or showing to the user a plan.

We will use `rich` for this. `rich` is a python library that allows adding style to terminal applications.

## Rich Primer

* More information can be found at the [docs](https://rich.readthedocs.io/en/latest/) or [repo](https://github.com/textualize/rich).
* `Console`: The main object used for interacting with the output. Imported from `rich.console`.
* Styling can be passed to the `print` method with the `style` keyword, or provided directly on the text using [bbcode](https://en.wikipedia.org/wiki/BBCode).
* One can combine non-conflicting styles within the same tag.
* One can close a group using `[/]` instead of rewritting the openning tag's text.


In [1]:
from rich.console import Console
console = Console()

#### Bold
```bbcode
[b]This text will be bold[/]
[bold]This text is also bold[/]
```

In [2]:
bold = "Plain [b]Bold[/] Text. Also [bold]bold[/]" 
console.print(bold)

Plain Bold Text. Also bold

#### URL
```bbcode
[link="https://en.wikipedia.org/wiki/BBCode]Learn More[/]
```

In [3]:
url = '[link="https://en.wikipedia.org/wiki/BBCode]Learn More[/]'
console.print(url)

]8;id=16410682;"https://en.wikipedia.org/wiki/BBCode\Learn More]8;;\

#### Colour

Rich supports adding colour to text. By default it supports all 256 [standard colours](https://rich.readthedocs.io/en/latest/appendix/colors.html#appendix-colors), which can be used as tags directly.

```bbcode
[red]Red text[/]
[dodger_blue1]Dodger Blue[/]
```

In [4]:
coloured = """
[red]Red text[/]
[dodger_blue1]Dodger Blue[/]
"""
console.print(coloured)

Red text
Dodger Blue

#### Tables

To create a table use the `Table` object from `rich.table`.

In [5]:
from rich.table import Table

t = Table(title="Some Title")
t.add_column("First Column", justify="right", style="cyan", no_wrap=True)
t.add_column("Second Column", justify="center", style="magenta")
t.add_row("First row", "$1,234.00")
t.add_row("Second row", "$5,678.90")
console.print(t)

           Some Title           
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ First Column ┃ Second Column ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│    First row │   $1,234.00   │
│   Second row │   $5,678.90   │
└──────────────┴───────────────┘

## Checklist Tools

In [6]:
def show(text):
    try:
        Console().print(text)
    except:
        print(text)

In [7]:
show("[red bold]Danger[/] Careful")

Danger Careful

In [8]:
checklist = []

In [9]:
def create_checklist(items: list[tuple[str, bool]]) -> bool:
    global checklist
    checklist = items
    return True

In [10]:
create_checklist([("Take pill at 09:00", False), ("Feed the cat", True)])

True

In [11]:
def add_task_to_checklist(task: str, done: bool = False) -> str:
    global checklist
    checklist.append((task, done))
    return "Success"

In [12]:
add_task_to_checklist("Wash sneakers")

'Success'

In [13]:
def get_checklist() -> list[tuple[str, bool]]:
    global checklist
    return checklist

In [14]:
get_checklist()

[('Take pill at 09:00', False),
 ('Feed the cat', True),
 ('Wash sneakers', False)]

In [15]:
def show_checklist():
    global checklist
    checklist_text = ""
    prefix = ""
    suffix = ""
    if checklist:
        prefix = "\nTasks Status:\n\n"
        suffix = "\n"
    for task, done in checklist:
        if done:
            checklist_text += f"\\[x] [green strike]{task}[/]\n"
        else:
            checklist_text += f"\\[ ] {task}\n"
    show(prefix + checklist_text + suffix)
    return checklist_text

In [16]:
show_checklist();

Tasks Status:

[ ] Take pill at 09:00
[x] Feed the cat
[ ] Wash sneakers

In [17]:
def update_checklist(index: int, done: bool) -> str:
    global checklist
    if len(checklist) - 1 < index:
        return "Index out of range"
    task, _ = checklist[index]
    checklist[index] = (task, done)
    return "Success"

## Setup Client

In [18]:
import os

from dotenv import load_dotenv
from openai import OpenAI
from providers import ANTHROPIC_API_URL, ANTHROPIC_API_MODEL

In [19]:
load_dotenv()

True

In [20]:
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")

In [21]:
client = OpenAI(api_key=ANTHROPIC_API_KEY, base_url=ANTHROPIC_API_URL)

In [22]:
add_task_to_checklist_json = {
    "name": "add_task_to_checklist",
    "description": (
        "Add a task to the checklist. Optionally set the status."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "task": {"type": "string", "description": "Short description of the task to accomplish"},
            "done": {"type": "boolean", "description": "Set the task status. Defaults to false (not done)."},
        },
        "additionalProperties": False,
        "required": ["task", "done"],
    }
}

update_checklist_json = {
    "name": "update_checklist",
    "description": (
        "Update the status of an item in the checklist by providing "
        "the position of the item and the new status. Returns True if updated, "
        "False otherwise (e.g. index out of bounds)."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "index": {"type": "integer", "description": "The index of the target task", "minimum": 0},
            "done": {"type": "boolean", "description": "Set the task status. Defaults to false (not done)."},
        },
        "additionalProperties": False,
        "required": ["index", "done"],
    }
}
show_checklist_json = {
    "name": "show_checklist",
    "description": (
        "Show task checklist usign pretty format. "
        "Returns the list of tasks and their respective status." 
    ),
}

In [23]:
tools = [
    {"type": "function", "function": add_task_to_checklist_json},
    {"type": "function", "function": update_checklist_json},
    {"type": "function", "function": show_checklist_json},
]

In [24]:
from typing import Any
import json


ALLOWED_TOOLS = {
    "add_task_to_checklist": add_task_to_checklist,
    "update_checklist": update_checklist,
    "show_checklist": show_checklist,
}


def simplify_history(history: list[dict[str, Any]]) -> [dict[str, str]]:
    return [{"role": h["role"], "content": h["content"]} for h in history]


def prepare_messages(
    text_message: str, system_prompt: str, history: list[dict[str, str]]
) -> list[dict[str, str]]:
    return (
        [
            {"role": "system", "content": system_prompt},
        ]
        + history
        + [{"role": "user", "content": text_message}]
    )


def call_llm(messages: list[dict[str, str]]):
    return client.chat.completions.create(
        model=ANTHROPIC_API_MODEL,
        messages=messages,
        tools=tools,
    )


def handle_call(messages: list[dict[str, str]]):
    response = call_llm(messages)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        messages.append(message)
        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            if tool_name in ALLOWED_TOOLS:
                args = json.loads(tool_call.function.arguments)
                result = ALLOWED_TOOLS[tool_name](**args)
            else:
                result = f"Error: Tool {tool_name} is not allowed."
            messages.append(
                {"role": "tool", "content": result, "tool_call_id": tool_call.id}
            )
        response = call_llm(messages)
    return response.choices[0].message.content


def chat_and_tools(message: str, history: list[dict[str, Any]] | None = None, system_prompt: str ="Your are a helpful agent"):
    if history is None:
        history = []
    history = simplify_history(history)
    messages = prepare_messages(message, system_prompt, history)
    return handle_call(messages)

In [25]:
create_checklist([])

True

In [26]:
show_checklist()

''

In [27]:
system_prompt = """
You are a maths tutor agent. Your task is to help solve mathematical
problems in a structured way, showing step by step the way to the solution.

When you are asked to solve a problem your first action is to create a
checklist of actions.

Then as you progress through the solution you will set the completed tasks
as done and show the state of the checklist.

Use the tools provided to create and display the
checklist status.

Remember to show the state every time there is a change.
"""

problem = """
There are two cars travelling at from two opposite locations.
Car A travels at 110 km/h and car B travels at 135 km/h.
The distance between them is 540 km.
If car A starts at 12:00pm and car B starts 30 mins after,
how far does car A travel before they cross paths?
"""

In [28]:
chat_and_tools(problem, system_prompt=system_prompt)

Tasks Status:

[ ] Understand the problem and identify given information
[ ] Set up equations for the distances traveled
[ ] Account for the 30-minute head start of Car A
[ ] Solve for the time when they meet
[ ] Calculate the distance traveled by Car A

Tasks Status:

[x] Understand the problem and identify given information
[ ] Set up equations for the distances traveled
[ ] Account for the 30-minute head start of Car A
[ ] Solve for the time when they meet
[ ] Calculate the distance traveled by Car A

Tasks Status:

[x] Understand the problem and identify given information
[x] Set up equations for the distances traveled
[x] Account for the 30-minute head start of Car A
[ ] Solve for the time when they meet
[ ] Calculate the distance traveled by Car A

Tasks Status:

[x] Understand the problem and identify given information
[x] Set up equations for the distances traveled
[x] Account for the 30-minute head start of Car A
[x] Solve for the time when they meet
[ ] Calculate the distance traveled by Car A

Tasks Status:

[x] Understand the problem and identify given information
[x] Set up equations for the distances traveled
[x] Account for the 30-minute head start of Car A
[x] Solve for the time when they meet
[x] Calculate the distance traveled by Car A

'## **Final Answer**\n\n**Car A travels approximately 272.76 km before they cross paths.**\n\n**Verification:**\n- Car B travels: 135 × (97/49) ≈ 267.24 km\n- Car A travels: 272.76 km\n- Total: 272.76 + 267.24 = 540 km ✓'